# BCT-AI-Pharma Results Visualization

Loads `results/simulation_results.csv` and `results/performance_metrics.json` (produced by `python main.py` or `python evaluation/run_simulation.py`) and visualizes them against the target metrics reported in the paper.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"

with open(RESULTS_DIR / "performance_metrics.json") as f:
    metrics = json.load(f)

sim_results = pd.read_csv(RESULTS_DIR / "simulation_results.csv")
sim_results.head()

## 1. Blockchain performance under varying transaction load (Table III)

In [ ]:
blockchain_df = pd.DataFrame(metrics["blockchain_performance"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(blockchain_df["tps_target"], blockchain_df["avg_latency_ms"], marker="o", label="Avg latency")
axes[0].plot(blockchain_df["tps_target"], blockchain_df["p95_latency_ms"], marker="s", label="P95 latency")
axes[0].set_xlabel("Offered load (TPS)")
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Latency vs. throughput")
axes[0].legend()

axes[1].plot(blockchain_df["tps_target"], blockchain_df["success_rate_pct"], marker="o", color="green")
axes[1].set_xlabel("Offered load (TPS)")
axes[1].set_ylabel("Success rate (%)")
axes[1].set_title("Transaction success rate vs. throughput")
plt.tight_layout()
plt.show()

## 2. LSTM demand forecasting / delay prediction vs. paper targets

In [ ]:
lstm_actual = metrics["lstm"]["metrics"]
lstm_target = metrics["lstm"]["target_metrics"]

lstm_compare = pd.DataFrame(
    {"metric": list(lstm_target.keys()),
     "paper_target": list(lstm_target.values()),
     "this_run": [lstm_actual[m] for m in lstm_target.keys()]}
)
lstm_compare.plot(x="metric", y=["paper_target", "this_run"], kind="bar", figsize=(7, 4))
plt.title("LSTM: paper target vs. this run")
plt.tight_layout()
plt.show()
lstm_compare

## 3. CNN packaging verification performance by tamper class

In [ ]:
cnn_actual = metrics["cnn"]["metrics"]
cnn_target = metrics["cnn"]["target_metrics"]

rows = []
for cls, target_vals in cnn_target.items():
    for metric_name, target_val in target_vals.items():
        rows.append({
            "class": cls,
            "metric": metric_name,
            "paper_target": target_val,
            "this_run": cnn_actual.get(cls, {}).get(metric_name, float("nan")),
        })
cnn_compare = pd.DataFrame(rows)

f1_compare = cnn_compare[cnn_compare["metric"] == "f1"]
plt.figure(figsize=(9, 4))
x = range(len(f1_compare))
plt.bar([i - 0.2 for i in x], f1_compare["paper_target"], width=0.4, label="Paper target")
plt.bar([i + 0.2 for i in x], f1_compare["this_run"], width=0.4, label="This run")
plt.xticks(list(x), f1_compare["class"], rotation=30, ha="right")
plt.ylabel("F1 score")
plt.title("CNN F1 by class: paper target vs. this run")
plt.legend()
plt.tight_layout()
plt.show()
cnn_compare

## 4. Isolation Forest cold chain anomaly detection by anomaly type

In [ ]:
if_actual = metrics["isolation_forest"]["metrics"]
if_target = metrics["isolation_forest"]["target_metrics"]

rows = []
for anomaly_type, target_vals in if_target.items():
    actual_vals = if_actual.get(anomaly_type, {})
    rows.append({
        "anomaly_type": anomaly_type,
        "target_detection": target_vals["detection"],
        "actual_detection": actual_vals.get("detection", float("nan")),
        "target_fpr": target_vals["fpr"],
        "actual_fpr": actual_vals.get("fpr", float("nan")),
    })
if_compare = pd.DataFrame(rows)

plt.figure(figsize=(9, 4))
x = range(len(if_compare))
plt.bar([i - 0.2 for i in x], if_compare["target_detection"], width=0.4, label="Paper target")
plt.bar([i + 0.2 for i in x], if_compare["actual_detection"], width=0.4, label="This run")
plt.xticks(list(x), if_compare["anomaly_type"], rotation=30, ha="right")
plt.ylabel("Detection rate")
plt.title("Isolation Forest detection rate by anomaly type")
plt.legend()
plt.tight_layout()
plt.show()
if_compare

## 5. Product Trust Score: distribution by drug class and weight sensitivity

In [ ]:
pts_scores = metrics["pts"]["sample_scores"]
pts_df = pd.DataFrame(pts_scores).T
pts_df.index.name = "drug_class"
pts_df

In [ ]:
# Nested {scenario: {component: [rows]}}; only "minor_excursion" / "temperature_compliance"
# is calibrated against the paper's one published sensitivity figure (Section 3.5.4,
# ~-0.08 PTS per +10-point weight shift) -- see pts/pts_sensitivity_analysis.py docstring.
scenario_curves = metrics["pts"]["sensitivity_curves"]
fig, axes = plt.subplots(1, len(scenario_curves), figsize=(6 * len(scenario_curves), 4.5), sharey=True)
for ax, (scenario, curves) in zip(axes, scenario_curves.items()):
    for component, rows in curves.items():
        df = pd.DataFrame(rows)
        ax.plot(df["delta_pct"], df["pts"], marker="o", label=component)
    ax.set_xlabel("Weight variation (percentage points)")
    ax.set_title(scenario.replace("_", " ").title())
    ax.legend(fontsize=8)
axes[0].set_ylabel("PTS")
fig.suptitle("Class A sensitivity: PTS vs. weight re-allocation, by excursion scenario")
plt.tight_layout()
plt.show()

## 6. Baseline comparison (Table VI)

In [ ]:
baseline_df = pd.DataFrame(metrics["baseline_comparison"]["table"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(baseline_df["method"], baseline_df["detection_rate_pct"], color="steelblue")
axes[0].set_ylabel("Detection rate (%)")
axes[0].set_title("Counterfeit detection rate by method")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(baseline_df["method"], baseline_df["localization_time_hours"], color="indianred")
axes[1].set_ylabel("Localization time (hours)")
axes[1].set_yscale("log")
axes[1].set_title("Recall/localization time by method (log scale)")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()
baseline_df

## 7. Statistical validation across independent runs

In [ ]:
sv = metrics["statistical_validation"]
print("Counterfeit detection paired t-test:", sv["counterfeit_detection_ttest"])
print("Recall efficiency paired t-test:", sv["recall_efficiency_ttest"])
print("95% CI (counterfeit detection):", sv["confidence_intervals_95"]["bct_ai_counterfeit_detection"])
print("95% CI (recall efficiency):", sv["confidence_intervals_95"]["bct_ai_recall_efficiency"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sim_results["run_id"], sim_results["counterfeit_detection_rate"], marker="o", label="Counterfeit detection")
ax.plot(sim_results["run_id"], sim_results["recall_efficiency"], marker="s", label="Recall efficiency")
ax.set_xlabel("Run")
ax.set_ylabel("Rate")
ax.set_title("Per-run metrics across 10 independent simulation runs")
ax.legend()
plt.tight_layout()
plt.show()